# MMF Data Preparation — Reproducibility Notebook
Auto-generated by mmf-agent after interactive `/prep-and-clean-data` session.

This notebook records all decisions made during the interactive data preparation
step and can be re-run to reproduce the same training table, scoring table, label
encoding and cleaning report from the source data.

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
source_table = "{source_table}"
unique_id_col = "{unique_id_col}"
ds_col = "{ds_col}"
y_col = "{y_col}"
freq = "{freq}"

# Covariates chosen in Step 5a, as Python lists ([] when unused).
static_features = {static_features}
dynamic_future_numerical = {dynamic_future_numerical}
dynamic_future_categorical = {dynamic_future_categorical}
dynamic_historical_numerical = {dynamic_historical_numerical}
dynamic_historical_categorical = {dynamic_historical_categorical}

# Step 6a: table holding the future regressor values ("" = no scoring table).
# The source table itself for option (a), another table for option (b).
scoring_source_table = "{scoring_source_table}"
scoring_unique_id_col = "{scoring_unique_id_col}"
scoring_ds_col = "{scoring_ds_col}"
# NULL fill per dynamic future column: "forward_fill", "mode", "mean", "drop", "none", or a number.
scoring_null_strategy = {scoring_null_strategy}

# Step 6b-ii: "label" encodes string covariates (option a); "drop" removes them (option c).
categorical_encoding = "{categorical_encoding}"

# Step 7b: "suggested" applies interpolation below 5% missing, forward fill from 5% up to
# the exclusion threshold, and excludes series above it. Any other value applies one method
# to every series: "interpolation", "forward_fill", "fill_zero", "drop_nulls" or "none".
imputation_method = "{imputation_method}"
exclusion_threshold = {exclusion_threshold}
iqr_multiplier = {iqr_multiplier}

In [ ]:
import pandas as pd
from pyspark.sql import types as T

source_fqn = f"{catalog}.{schema}.{source_table}"
train_table = f"{catalog}.{schema}.{use_case}_train_data"
scoring_table = f"{catalog}.{schema}.{use_case}_scoring_data"
encoding_table = f"{catalog}.{schema}.{use_case}_label_encoding"
report_table = f"{catalog}.{schema}.{use_case}_cleaning_report"


def unique(cols):
    return list(dict.fromkeys(cols))


numerical_cols = unique(dynamic_future_numerical + dynamic_historical_numerical)
categorical_cols = unique(static_features + dynamic_future_categorical + dynamic_historical_categorical)
covariate_cols = unique(static_features + dynamic_future_numerical + dynamic_future_categorical
                        + dynamic_historical_numerical + dynamic_historical_categorical)
future_cols = unique(dynamic_future_numerical + dynamic_future_categorical)
has_scoring = bool(scoring_source_table) and bool(future_cols)
scoring_fqn = f"{catalog}.{schema}.{scoring_source_table}" if has_scoring else None

print(f"Source: {source_fqn}")
src = spark.sql(f"""
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT {unique_id_col}) AS n_series,
       MIN({ds_col}) AS min_ds, MAX({ds_col}) AS max_ds
FROM {source_fqn}""").first()
print(f"  Rows: {src['n_rows']:,}, series: {src['n_series']:,}, dates: {src['min_ds']} to {src['max_ds']}")
print(f"  Frequency: {freq}; mapping: unique_id={unique_id_col}, ds={ds_col}, y={y_col}")
print(f"  Covariates: {covariate_cols or 'none (univariate)'}")
print(f"  Scoring source: {scoring_fqn or 'none'}")

## Step 6: Create the training table (with covariates) and the scoring table

In [ ]:
def aligned_ds(col):
    return {
        "H": f"CAST({col} AS TIMESTAMP)",
        "D": f"CAST({col} AS DATE)",
        "W": f"CAST(DATE_TRUNC('week', {col}) + INTERVAL 6 DAY AS DATE)",
        "M": f"CAST(LAST_DAY({col}) AS DATE)",
    }[freq]


def aligned_select(table, id_c, date_c, cols, target=None, where="TRUE"):
    """One row per (unique_id, aligned ds). Weekly and monthly data are aggregated:
    SUM for the target and numerical covariates, MAX for categorical and static ones."""
    agg = freq in ("W", "M")
    exprs = [f"CAST({id_c} AS STRING) AS unique_id", f"{aligned_ds(date_c)} AS ds"]
    if target:
        exprs.append(f"SUM(CAST({target} AS DOUBLE)) AS y" if agg else f"CAST({target} AS DOUBLE) AS y")
    for c in cols:
        exprs.append((f"SUM({c})" if c in numerical_cols else f"MAX({c})") + f" AS {c}" if agg else c)
    sql = f"SELECT {', '.join(exprs)} FROM {table} WHERE {where}"
    if agg:
        sql += f" GROUP BY CAST({id_c} AS STRING), {aligned_ds(date_c)}"
    return sql


spark.sql(f"CREATE OR REPLACE TABLE {train_table} AS "
          + aligned_select(source_fqn, unique_id_col, ds_col, covariate_cols, y_col, f"{y_col} IS NOT NULL"))

# Per-series row counts before cleaning, kept in pandas because the table is replaced below.
original_counts = spark.sql(f"SELECT unique_id, COUNT(*) AS original_count FROM {train_table} GROUP BY unique_id").toPandas()
print(f"Created {train_table}: {int(original_counts['original_count'].sum()):,} rows, {len(original_counts):,} series")

if has_scoring:
    static_join = ""
    static_select = ""
    if static_features:
        static_select = "".join(f", t.{c}" for c in static_features)
        static_join = (f"JOIN (SELECT unique_id, {', '.join(f'MAX({c}) AS {c}' for c in static_features)} "
                       f"FROM {train_table} GROUP BY unique_id) t ON s.unique_id = t.unique_id")
    spark.sql(f"""
    CREATE OR REPLACE TABLE {scoring_table} AS
    SELECT s.unique_id, s.ds{static_select}{''.join(f', s.{c}' for c in future_cols)}
    FROM ({aligned_select(scoring_fqn, scoring_unique_id_col, scoring_ds_col, future_cols)}) s
    {static_join}
    WHERE s.ds > (SELECT MAX(ds) FROM {train_table})
    """)
    sc = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT unique_id) AS s, MIN(ds) AS lo, MAX(ds) AS hi FROM {scoring_table}").first()
    print(f"Created {scoring_table}: {sc['n']:,} rows, {sc['s']:,} series, {sc['lo']} to {sc['hi']}")

In [ ]:
alignment_check = {
    "M": "ds <> LAST_DAY(ds)",
    "W": "ds <> DATE_TRUNC('week', ds) + INTERVAL 6 DAY",
}.get(freq)

if alignment_check:
    for t in [train_table] + ([scoring_table] if has_scoring else []):
        bad = spark.sql(f"SELECT COUNT(*) AS n FROM {t} WHERE {alignment_check}").first()["n"]
        assert bad == 0, f"Date alignment failed: {bad} rows in {t} violate {alignment_check}"
print(f"Date alignment check (freq={freq}): PASS")

## Step 6a: Validate the scoring table and fill NULL future regressors

In [ ]:
if has_scoring:
    missing_series = spark.sql(f"""
    SELECT COUNT(*) AS n FROM (
      SELECT DISTINCT unique_id FROM {train_table}
      EXCEPT SELECT DISTINCT unique_id FROM {scoring_table})""").first()["n"]
    print(f"Training series without scoring rows: {missing_series}")

    def null_counts():
        row = spark.sql(f"SELECT {', '.join(f'COUNT_IF({c} IS NULL) AS {c}' for c in future_cols)} FROM {scoring_table}").first()
        return row.asDict()

    print(f"NULLs before filling: {null_counts()}")
    for c in future_cols:
        rule = scoring_null_strategy.get(c, "none")
        if rule == "none":
            continue
        if rule == "forward_fill":
            # Each series' last known value from the training table.
            spark.sql(f"""
            MERGE INTO {scoring_table} AS s
            USING (SELECT unique_id, MAX_BY({c}, ds) AS v FROM {train_table}
                   WHERE {c} IS NOT NULL GROUP BY unique_id) AS l
            ON s.unique_id = l.unique_id
            WHEN MATCHED AND s.{c} IS NULL THEN UPDATE SET {c} = l.v""")
        elif rule == "drop":
            spark.sql(f"DELETE FROM {scoring_table} WHERE {c} IS NULL")
        else:
            if rule == "mode":
                value = spark.sql(f"SELECT {c} FROM {train_table} WHERE {c} IS NOT NULL "
                                  f"GROUP BY {c} ORDER BY COUNT(*) DESC, {c} LIMIT 1").first()[0]
            elif rule == "mean":
                value = spark.sql(f"SELECT AVG({c}) FROM {train_table}").first()[0]
            else:
                value = rule
            spark.sql(f"UPDATE {scoring_table} SET {c} = :v WHERE {c} IS NULL", args={"v": value})
    remaining = null_counts()
    print(f"NULLs after filling: {remaining}")
    if any(remaining.values()):
        print("WARNING: NULL future regressors remain; models that use them may fail.")
else:
    print("No scoring table (no dynamic future regressors).")

## Step 6b: Encode categorical covariates and cast every covariate to DOUBLE

In [ ]:
tables = [train_table] + ([scoring_table] if has_scoring else [])
types = {f.name: f.dataType for f in spark.table(train_table).schema.fields}
numeric_types = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
string_cols = [c for c in categorical_cols if not isinstance(types[c], numeric_types)]
encoded_cols = []

if string_cols and categorical_encoding == "drop":
    for t in tables:
        keep = [c for c in spark.table(t).columns if c not in string_cols]
        spark.sql(f"CREATE OR REPLACE TABLE {t} AS SELECT {', '.join(keep)} FROM {t}")
    covariate_cols = [c for c in covariate_cols if c not in string_cols]
    print(f"Dropped string covariates: {string_cols}. Remove them from the forecasting notebooks' covariate lists.")
elif string_cols:
    parts = []
    for c in string_cols:
        sources = [f"SELECT DISTINCT CAST({c} AS STRING) AS v FROM {t} WHERE {c} IS NOT NULL"
                   for t in tables if c in spark.table(t).columns]
        parts.append(f"SELECT '{c}' AS column_name, v AS original_value, "
                     f"DENSE_RANK() OVER (ORDER BY v) - 1 AS encoded_value FROM ({' UNION '.join(sources)})")
    spark.sql(f"CREATE OR REPLACE TABLE {encoding_table} AS {' UNION ALL '.join(parts)}")
    encoded_cols = string_cols

for t in tables:
    exprs, joins = [], []
    for i, c in enumerate(spark.table(t).columns):
        if c in encoded_cols:
            exprs.append(f"CAST(e{i}.encoded_value AS DOUBLE) AS {c}")
            joins.append(f"LEFT JOIN (SELECT original_value, encoded_value FROM {encoding_table} "
                         f"WHERE column_name = '{c}') e{i} ON CAST(t.{c} AS STRING) = e{i}.original_value")
        elif c in covariate_cols:
            exprs.append(f"CAST(t.{c} AS DOUBLE) AS {c}")
        else:
            exprs.append(f"t.{c}")
    spark.sql(f"CREATE OR REPLACE TABLE {t} AS SELECT {', '.join(exprs)} FROM {t} t {' '.join(joins)}")

if encoded_cols:
    print("Label encoding (mapping saved to " + encoding_table + "):")
    spark.sql(f"SELECT column_name, COUNT(*) AS n_values, MAX(encoded_value) AS max_code "
              f"FROM {encoding_table} GROUP BY column_name ORDER BY column_name").show(truncate=False)
print(f"All covariates are DOUBLE in: {tables}")

## Step 7: Missing data assessment and imputation

In [ ]:
interval = {"H": "INTERVAL 1 HOUR", "D": "INTERVAL 1 DAY", "W": "INTERVAL 7 DAY"}.get(freq)

# Monthly spine must be drift-safe: SEQUENCE(month_end, ..., INTERVAL 1 MONTH) drifts off
# true month-ends for series whose first month is a 30/28-day month. Sequence on month
# STARTs then wrap in LAST_DAY so every spine date is the true month-end.
if freq == "M":
    date_spine_cte = (
        "date_spine AS (\n"
        "  SELECT unique_id, LAST_DAY(m) AS expected_ds FROM (\n"
        "    SELECT s.unique_id,\n"
        "           EXPLODE(SEQUENCE(TRUNC(s.min_ds,'MM'), TRUNC(g.max_ds,'MM'), INTERVAL 1 MONTH)) AS m\n"
        "    FROM series_bounds s CROSS JOIN global_max g\n"
        "  )\n"
        ")"
    )
else:
    date_spine_cte = (
        "date_spine AS (\n"
        f"  SELECT s.unique_id, EXPLODE(SEQUENCE(s.min_ds, g.max_ds, {interval})) AS expected_ds\n"
        "  FROM series_bounds s CROSS JOIN global_max g\n"
        ")"
    )
spine_ctes = f"""
WITH global_max AS (SELECT MAX(ds) AS max_ds FROM {train_table}),
series_bounds AS (SELECT unique_id, MIN(ds) AS min_ds FROM {train_table} GROUP BY unique_id),
{date_spine_cte}"""

# Kept in pandas: the training table is replaced by the backfill below.
missing = spark.sql(f"""
{spine_ctes},
spine_joined AS (
  SELECT sp.unique_id, t.y
  FROM date_spine sp
  LEFT JOIN {train_table} t ON sp.unique_id = t.unique_id AND sp.expected_ds = t.ds
)
SELECT unique_id,
  COUNT(*) AS expected_count,
  SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) AS missing_count,
  ROUND(SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS missing_pct
FROM spine_joined GROUP BY unique_id
""").toPandas()


def series_method(row):
    if row.missing_count == 0:
        return "none"
    if exclusion_threshold > 0 and row.missing_pct > exclusion_threshold:
        return "excluded"
    if imputation_method == "suggested":
        return "interpolation" if row.missing_pct < 5 else "forward_fill"
    return imputation_method


missing["imputation_method"] = missing.apply(series_method, axis=1)
spark.createDataFrame(missing[["unique_id", "imputation_method"]]).createOrReplaceTempView("series_method")

trailing = spark.sql(f"""
SELECT COUNT(*) AS n FROM (
  SELECT unique_id, MAX(ds) AS last_ds FROM {train_table} WHERE y IS NOT NULL GROUP BY unique_id
) WHERE last_ds < (SELECT MAX(ds) FROM {train_table})""").first()["n"]

print("Missing data summary:")
print(f"  Complete (no gaps): {(missing.missing_count == 0).sum()}")
print(f"  Low (< 5%):         {((missing.missing_pct > 0) & (missing.missing_pct < 5)).sum()}")
print(f"  Medium (5-20%):     {((missing.missing_pct >= 5) & (missing.missing_pct <= 20)).sum()}")
print(f"  Above {exclusion_threshold}%:        {(missing.missing_pct > exclusion_threshold).sum()}")
print(f"  Trailing gaps:      {trailing} series end before the global max date")
print("Per-series method:")
print(missing["imputation_method"].value_counts().to_string())

In [ ]:
# Backfill the spine; covariates on new rows are NULL until Step 7d.
cov_select = "".join(f", t.{c}" for c in covariate_cols)
spark.sql(f"""
CREATE OR REPLACE TABLE {train_table} AS
{spine_ctes}
SELECT sp.unique_id, sp.expected_ds AS ds, t.y{cov_select}
FROM date_spine sp
LEFT JOIN {train_table} t ON sp.unique_id = t.unique_id AND sp.expected_ds = t.ds
""")

spark.sql(f"DELETE FROM {train_table} WHERE unique_id IN "
          "(SELECT unique_id FROM series_method WHERE imputation_method = 'excluded')")

by_series = "PARTITION BY unique_id ORDER BY ds"
ffill = f"LAST_VALUE(y) IGNORE NULLS OVER ({by_series} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)"
# Interpolation has no later neighbour on trailing gaps, so it falls back to forward fill.
interp = (f"COALESCE((LAG(y) IGNORE NULLS OVER ({by_series}) + LEAD(y) IGNORE NULLS OVER ({by_series})) / 2, {ffill})")
spark.sql(f"""
CREATE OR REPLACE TABLE {train_table} AS
SELECT unique_id, ds,
  CASE
    WHEN y IS NOT NULL THEN y
    WHEN imputation_method = 'interpolation' THEN {interp}
    WHEN imputation_method = 'forward_fill' THEN {ffill}
    WHEN imputation_method = 'fill_zero' THEN 0D
  END AS y{''.join(f', {c}' for c in covariate_cols)}
FROM (SELECT * FROM {train_table} JOIN series_method USING (unique_id))
""")
if imputation_method == "drop_nulls":
    spark.sql(f"DELETE FROM {train_table} WHERE y IS NULL")

remaining = spark.sql(f"SELECT unique_id, COUNT_IF(y IS NULL) AS remaining_nulls FROM {train_table} GROUP BY unique_id").toPandas()
n_remaining = int(remaining["remaining_nulls"].sum())
print(f"Imputation complete; remaining NULLs in y: {n_remaining:,}")
if n_remaining:
    print("WARNING: some y values are still NULL; exclude those series or switch them to forward fill.")

In [ ]:
# Step 7d: forward fill, then back fill, every covariate on the backfilled rows.
if covariate_cols:
    fills = "".join(
        f""", COALESCE(
            LAST_VALUE({c}) IGNORE NULLS OVER ({by_series} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
            FIRST_VALUE({c}) IGNORE NULLS OVER ({by_series} ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING)
          ) AS {c}"""
        for c in covariate_cols)
    spark.sql(f"CREATE OR REPLACE TABLE {train_table} AS SELECT unique_id, ds, y{fills} FROM {train_table}")
    cov_nulls = spark.sql(f"SELECT {', '.join(f'COUNT_IF({c} IS NULL) AS {c}' for c in covariate_cols)} "
                          f"FROM {train_table} WHERE y IS NOT NULL").first().asDict()
    if any(cov_nulls.values()):
        print(f"WARNING: covariate NULLs remain on rows with y: {cov_nulls}")
    else:
        print("Training covariate NULL check: PASS (0 nulls across all covariate columns)")
else:
    print("No covariates (univariate run); skipping covariate imputation.")

## Step 8: Anomaly detection and capping

In [ ]:
iqr_stats = f"""
SELECT unique_id, PERCENTILE(y, 0.25) AS q1, PERCENTILE(y, 0.75) AS q3,
       PERCENTILE(y, 0.75) - PERCENTILE(y, 0.25) AS iqr
FROM {train_table} GROUP BY unique_id"""
multiplier = iqr_multiplier if iqr_multiplier > 0 else 1.5

# Counted before capping and kept in pandas, because capping rewrites the table.
anomalies = spark.sql(f"""
SELECT t.unique_id, COUNT(*) AS total_points,
       COUNT_IF(t.y < s.q1 - {multiplier} * s.iqr OR t.y > s.q3 + {multiplier} * s.iqr) AS anomaly_count
FROM {train_table} t JOIN ({iqr_stats}) s ON t.unique_id = s.unique_id
GROUP BY t.unique_id""").toPandas()
affected = anomalies[anomalies.anomaly_count > 0]
print(f"Anomaly detection ({multiplier}x IQR): {len(affected)} / {len(anomalies)} series, "
      f"{int(affected.anomaly_count.sum()):,} points")

if iqr_multiplier > 0:
    spark.sql(f"""
    MERGE INTO {train_table} AS t
    USING ({iqr_stats}) AS s
    ON t.unique_id = s.unique_id
    WHEN MATCHED AND (t.y < s.q1 - {multiplier} * s.iqr OR t.y > s.q3 + {multiplier} * s.iqr) THEN UPDATE SET
      y = CASE WHEN t.y < s.q1 - {multiplier} * s.iqr THEN s.q1 - {multiplier} * s.iqr
               ELSE s.q3 + {multiplier} * s.iqr END""")
    anomalies["anomalies_capped"] = anomalies["anomaly_count"]
    print(f"Capped {int(anomalies.anomalies_capped.sum()):,} values at {iqr_multiplier}x IQR")
else:
    anomalies["anomalies_capped"] = 0
    print("Anomaly capping skipped (iqr_multiplier = 0)")

In [ ]:
# Step 8d: drop scoring series that are no longer in the training table.
if has_scoring:
    spark.sql(f"DELETE FROM {scoring_table} WHERE unique_id NOT IN (SELECT DISTINCT unique_id FROM {train_table})")
    orphans = spark.sql(f"""
    SELECT COUNT(*) AS n FROM (SELECT unique_id FROM {scoring_table} EXCEPT SELECT unique_id FROM {train_table})""").first()["n"]
    assert orphans == 0, f"{orphans} scoring series are not in {train_table}"
    print("Scoring/train series reconciliation: PASS (scoring_not_in_train=0)")

## Step 9: Cleaning report

In [ ]:
final_counts = spark.sql(f"SELECT unique_id, COUNT(*) AS final_count FROM {train_table} GROUP BY unique_id").toPandas()
report = (original_counts
          .merge(missing[["unique_id", "missing_count", "missing_pct", "imputation_method"]], on="unique_id", how="left")
          .merge(final_counts, on="unique_id", how="left")
          .merge(remaining, on="unique_id", how="left")
          .merge(anomalies[["unique_id", "anomalies_capped"]], on="unique_id", how="left"))
report["excluded"] = report["imputation_method"] == "excluded"
for c in ["final_count", "remaining_nulls", "anomalies_capped", "missing_count"]:
    report[c] = report[c].fillna(0).astype("int64")
report["missing_filled"] = (report["missing_count"] - report["remaining_nulls"]).where(~report["excluded"], 0)
report["iqr_multiplier"] = float(iqr_multiplier)
report["exclusion_reason"] = [
    f"missing {p}% > {exclusion_threshold}% threshold" if e else None
    for p, e in zip(report["missing_pct"], report["excluded"])]

schema_ = T.StructType([
    T.StructField("unique_id", T.StringType()), T.StructField("original_count", T.LongType()),
    T.StructField("final_count", T.LongType()), T.StructField("missing_filled", T.LongType()),
    T.StructField("imputation_method", T.StringType()), T.StructField("anomalies_capped", T.LongType()),
    T.StructField("iqr_multiplier", T.DoubleType()), T.StructField("excluded", T.BooleanType()),
    T.StructField("exclusion_reason", T.StringType())])
cols = [f.name for f in schema_.fields]
(spark.createDataFrame(report[cols].astype(object).where(report[cols].notna(), None).values.tolist(), schema_)
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(report_table))

tr = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT unique_id) AS s, MIN(ds) AS lo, MAX(ds) AS hi FROM {train_table}").first()
print(f"Data preparation complete for '{use_case}'")
print(f"  Training table: {train_table} ({tr['n']:,} rows, {tr['s']:,} series, {tr['lo']} to {tr['hi']})")
if has_scoring:
    sc = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT unique_id) AS s FROM {scoring_table}").first()
    print(f"  Scoring table:  {scoring_table} ({sc['n']:,} rows, {sc['s']:,} series)")
if encoded_cols:
    print(f"  Label encoding: {encoding_table}")
print(f"  Cleaning report: {report_table}")
print(f"  Values imputed: {int(report.missing_filled.sum()):,}; series excluded: {int(report.excluded.sum())}; "
      f"values capped: {int(report.anomalies_capped.sum()):,}")